# sele_pdb and all of its helper functions

In [5]:
from pathlib import Path
import sys
sys.path.append( str( Path("../../../.." ).resolve()) )

- ## sele_res_idx()

In [8]:
# load test pdbs

import gemmi
ev2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a.pdb")
cox2a_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/coxb4_2a.pdb")
ev2a1_pdb = gemmi.read_pdb("../../../../tests/testdata/protein/ev2a_1.pdb")
from xaidar.data.protocols import protein_processing
from xaidar.data.molecModels import sele_pdb, sele_AA, sele_model
from xaidar.data.molecModels import get_pdb_stats
ev2a_prot = protein_processing(ev2a_pdb)
ev2a1_prot = protein_processing(ev2a1_pdb)
cox2a_prot = cox2a_pdb
for foo in [ sele_AA, sele_model]:
    cox2a_prot = sele_pdb(cox2a_prot, foo)
# get_pdb_stats(ev2a_prot)
# get_pdb_stats(ev2a1_prot)
# get_pdb_stats(cox2a_prot)

In [9]:
import gemmi
from xaidar.data.molecModels import sele_pdb

def sele_res_idx( lst_res: list[gemmi.Residue], lst_slices: list[ tuple ],
                slice_within = True, level = False  ):
    """
    Remove residues from a list based on specified slices.
    Args:
    - lst_res (list[gemmi.Residue]): List of residue objects.
    - lst_slices (list[tuple]): List of tuples specifying slices to remove.
        Must be in (start, end) format, where 'start' is inclusive and 'end' is exclusive.
        The slices should not overlap. The integers must be positive and within the range of lst_res.
    - slice_within (bool, optional): If True, remove residues within the slices.
        If False, remove residues outside the slices. Defaults to True.
    - level (bool, optional): If True, operate at residue level. Defaults to False.
    Returns:
    - list[gemmi.Residue]: Updated list of residue objects after removal.
    """
    if level: return "residue"
    lst_slices = sorted( lst_slices, reverse=True )
    new_lst_res = lst_res if slice_within else []
    for slice_idx in lst_slices:
        start, end = slice_idx
        if slice_within:
            del new_lst_res[start:end]
        else:
            new_lst_res = new_lst_res + lst_res[start:end]
    return new_lst_res

In [14]:
# Test
from xaidar.data.molecModels import get_chain_seq, model_seqAlign

from copy import deepcopy
refprot = ev2a_prot
queryprot = cox2a_prot
gap_query = deepcopy( queryprot)
gap_query_seq = get_chain_seq( gap_query )[0]
print( gap_query_seq )

gap_query = sele_pdb( gap_query, sele_res_idx, [  (10,20) ] ) # testing removing residues 10-19
gap_query_seq = get_chain_seq( gap_query )[0]
print( gap_query_seq )

alignment = model_seqAlign( refprot, queryprot,).visualize().map_matching_res( match_type = "exact", gaps = True)
print( "\t###########################################################")
alignment = model_seqAlign( refprot, gap_query,).visualize().map_matching_res( match_type = "exact", gaps = True)



GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
GPYGHQSGAVHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
                |         |         |         |         *         |         |         |         |         +         |         |         |         |         *
Ref:   ------SGAIYVGNYRVVNRHLATHNDWANLVWEDSSRDLLVSSTTAQGCDTIARCDCQTGVYYCSSRRKHYPVSFSKPSLIFVEASEYYPARYQSHLMLAVGHSEPGDCGGILRCQHGVVGIVSTGGNGLVGFADVRDLLWLD----EE
             |||:|||||:|||||||||.||.|.||||.:||||||:|||.||||||||.|.||||:|:|:.|||||||..|.|:.|:.|||||.|||||::||.|.|||||.||||||:|||:|:|:.||.|:||||||||||||:    |:
Query: GPYGHQSGAVYVGNYKVVNRHLATHVDWQNCVWEDYNRDLLVSTTTAHGCDTIARCQCTTGVYFCASKSKHYPVSFEGPGLVEVQESEYYPKRYQSHVLLATGFSEPGDAGGILRCEHGVIGLVTMGGEGVVGFADVRDLLWLEDDAMEQ
	###########################################################
               